In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 07 - Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e Inteligência Artificial?
# ---------------------------------------------------------------------

"""
A Gold 07 reúne indicadores que podem apoiar a identificação de oportunidades e desafios sob diferentes perspectivas, como gestão, atração e retenção de profissionais e barreiras para adoção de IA. Antes de consolidar essas informações em uma leitura executiva, a estrutura é inspecionada para validar quais categorias e períodos estão efetivamente disponíveis.
"""
caminho_gold_07 = (
    PROJECT_ROOT
    / "Gold"
    / "perguntas_negocio"
    / "gold_07_oportunidades_desafios"
)

arquivos_gold_07 = [
    str(arquivo) for arquivo in caminho_gold_07.glob("part-*.csv")
]

print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_07)

if not arquivos_gold_07:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_07}"
    )

df_gold_07 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_07)
)


# ---------------------------------------------------------------------
# INSPEÇÃO INICIAL DA GOLD 07
# ---------------------------------------------------------------------

"""
A inspeção inicial confirma a granularidade da Gold antes das análises. O notebook executado apresenta 122 registros e seis colunas, organizados por edição, opção, quantidade de elegíveis, quantidade de seleções, percentual de adoção e categoria.
"""
print("\n" + "=" * 100)
print("1. INSPEÇÃO INICIAL DA GOLD 07")
print("=" * 100)

df_gold_07.show(50, truncate=False)

print("\nSCHEMA:")
df_gold_07.printSchema()

print("\nQuantidade de linhas:", df_gold_07.count())
print("Quantidade de colunas:", len(df_gold_07.columns))

print("\nCOLUNAS:")

for coluna in df_gold_07.columns:
    print("-", coluna)


# ---------------------------------------------------------------------
# QUANTIDADE DE VALORES DISTINTOS POR COLUNA
# ---------------------------------------------------------------------

"""
A quantidade de valores distintos ajuda a dimensionar a variedade disponível em cada campo. Os outputs confirmam três edições, 36 opções e quatro categorias analíticas, mostrando que a Gold consolida diferentes indicadores em uma estrutura comum de adoção.
"""
print("\n" + "=" * 100)
print("2. QUANTIDADE DE VALORES DISTINTOS POR COLUNA")
print("=" * 100)

qtd_distintos = (
    df_gold_07
    .agg(
        *[
            F.countDistinct(F.col(coluna)).alias(coluna)
            for coluna in df_gold_07.columns
        ]
    )
)

qtd_distintos.show(truncate=False)


# ---------------------------------------------------------------------
# VALORES NULOS POR COLUNA
# ---------------------------------------------------------------------

"""
A validação não identificou valores nulos em nenhuma das seis colunas. Dessa forma, os registros possuem os campos necessários para identificar edição, categoria, opção e respectivas métricas, sem necessidade de tratamento de ausência nesta etapa.
"""
print("\n" + "=" * 100)
print("3. VALORES NULOS POR COLUNA")
print("=" * 100)

nulos = (
    df_gold_07
    .agg(
        *[
            F.sum(
                F.when(F.col(coluna).isNull(), 1).otherwise(0)
            ).alias(coluna)
            for coluna in df_gold_07.columns
        ]
    )
)

nulos.show(truncate=False)


# ---------------------------------------------------------------------
# LINHAS DUPLICADAS
# ---------------------------------------------------------------------

"""
A checagem de duplicidade garante que uma mesma combinação de informações não esteja sendo contabilizada mais de uma vez. O notebook confirma 122 linhas únicas entre os 122 registros, sem duplicidades na Gold 07.
"""
print("\n" + "=" * 100)
print("4. VALIDAÇÃO DE DUPLICIDADES")
print("=" * 100)

total_linhas = df_gold_07.count()
total_unicas = df_gold_07.dropDuplicates().count()
total_duplicadas = total_linhas - total_unicas

print("Total de linhas:", total_linhas)
print("Total de linhas únicas:", total_unicas)
print("Total de linhas duplicadas:", total_duplicadas)


# ---------------------------------------------------------------------
# INSPEÇÃO DAS PRINCIPAIS DIMENSÕES
# ---------------------------------------------------------------------

"""
A inspeção é preparada para diferentes estruturas possíveis da Gold, verificando somente as colunas que realmente existem. Nesta versão, os principais eixos disponíveis são edicao e categoria.

Os outputs identificam quatro categorias: critérios para escolher emprego, desafios como gestor, motivos de insatisfação profissional e motivos para não usar IA. Esses grupos permitem organizar oportunidades e desafios sem misturar indicadores com significados diferentes.
"""
print("\n" + "=" * 100)
print("5. INSPEÇÃO DAS PRINCIPAIS DIMENSÕES")
print("=" * 100)

colunas_inspecao = [
    "edicao",
    "variavel",
    "dimensao",
    "categoria",
    "tipo",
    "indicador"
]

for coluna in colunas_inspecao:
    if coluna in df_gold_07.columns:
        print(f"\nVALORES EXISTENTES EM '{coluna}':")

        (
            df_gold_07
            .select(coluna)
            .distinct()
            .orderBy(coluna)
            .show(200, truncate=False)
        )


# ---------------------------------------------------------------------
# VISUALIZAÇÃO COMPLETA DA GOLD 07
# ---------------------------------------------------------------------

"""
A visualização completa é necessária para avaliar a cobertura histórica de cada categoria antes de criar comparações entre edições. Os outputs mostram que nem todos os indicadores possuem a mesma disponibilidade ao longo das três pesquisas.

Por exemplo, critérios para escolher emprego aparecem em 2024-2025 e 2025-2026, enquanto desafios de gestores e barreiras para adoção de IA possuem registros nas três edições. Também existem opções com percentual igual a zero em 2023-2024, portanto esses casos devem ser avaliados antes de interpretar o zero automaticamente como ausência do desafio.
"""
print("\n" + "=" * 100)
print("6. CONTEÚDO DA GOLD 07")
print("=" * 100)

colunas_ordenacao = [
    coluna
    for coluna in [
        "edicao",
        "variavel",
        "dimensao",
        "categoria",
        "tipo",
        "indicador"
    ]
    if coluna in df_gold_07.columns
]

if colunas_ordenacao:
    (
        df_gold_07
        .orderBy(*colunas_ordenacao)
        .show(200, truncate=False)
    )

else:
    df_gold_07.show(
        200,
        truncate=False
    )